### Exercise 9 & 10: Hypothesis Testing (Means) — large vs small samples — with `statsmodels`

This notebook teaches **hypothesis testing for means** using the standard methods taught in class:

1. One-sample mean test (large sample, one-sided)
2. One-sample mean test (large sample, two-sided)
3. One-sample mean test (small sample, one-sided)
4. One-sample mean test (small sample, two-sided)
5. Difference of two large-sample means (one-sided)
6. Difference of two small-sample means (two-sided) 

We will use:
- **NHANES 2015–2016** (CDC/NCHS) for Tasks 1–5 (public health context)
- **RAND Health Insurance Experiment** (`statsmodels` dataset) for Task 6


In [1]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
from statsmodels.stats.weightstats import DescrStatsW, ztest, ttest_ind

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


##### Part A —  Load NHANES 2015–2016 (for Tasks 1–5)

NHANES is a large, public, high-integrity health survey run by the CDC/NCHS.

We will work with:
- `sys_mean`: average systolic blood pressure (mm Hg) from multiple readings
- `RIAGENDR`: sex (1 = Male, 2 = Female)

We’ll build `sys_mean` from the available systolic BP readings.

*(NHANES files are public and hosted by CDC. The code below downloads XPT files.)*


In [2]:
# Key: SEQN (Respondent sequence number)
base = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/"

# Demographics
demo = pd.read_sas(base + "DEMO_I.XPT")

# Blood pressure
bpx = pd.read_sas(base + "BPX_I.XPT")
df = demo.merge(bpx, on="SEQN", how="left")

# Create systolic BP mean from up to 4 readings
sys_cols = ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]
df["sys_mean"] = df[sys_cols].mean(axis=1, skipna=True)

df[["SEQN", "RIAGENDR", "sys_mean"]].head()

,SEQN,RIAGENDR,sys_mean
0,83732.0,1.0,122.666667
1,83733.0,1.0,140.000000
2,83734.0,1.0,135.333333
3,83735.0,2.0,134.000000
4,83736.0,2.0,104.000000


##### ✅ Task for students:
Drop missing of `sys_mean` -> cast it to float -> and convert it to a numpy array called `x_large`.

(We’ll use this full sample for the “large sample” tests.)

In [3]:
# Your code here
x_large = df["sys_mean"].dropna().astype(float).to_numpy()
x_large[:10], x_large.shape

(array([122.66666667, 140.        , 135.33333333, 134.        ,
        104.        , 119.33333333, 100.        , 111.33333333,
        118.        , 179.33333333]),
 (7363,))

##### ✅ Task for students: Hypothesis testing of large-sample mean 

We want to test whether **average systolic BP** is **higher than 120 mm Hg**.

Set up:
- **H0:** μ = 120  
- **H1:** μ > 120  

Using a **large-sample mean test (normal / z test)** with `DescrStatsW`:

1) define wheather it is one-sided or two-sided.
2) compute the test statistic  
3) compute the p-value  
4) state your conclusion at α = 0.05


In [4]:
# Your answer here
Answer = "it is one sided since the alternative hypothesis is μ > 120"


# Your code here ( please use ztest_mean from DescrStatsW)
mu0 = 120

ds = DescrStatsW(x_large)
z_stat, p_value = ds.ztest_mean(value=mu0, alternative="larger")

pd.DataFrame([{
    "n": int(ds.nobs),
    "sample_mean": float(ds.mean),
    "mu0": mu0,
    "test": "one-sample z test (one-sided)",
    "alternative": "mean > mu0",
    "z_stat": float(z_stat),
    "p_value": float(p_value),
    "alpha": 0.05,
    "decision": "Reject H0" if p_value < 0.05 else "Fail to reject H0",

}])

,n,sample_mean,mu0,test,alternative,z_stat,p_value,alpha,decision
0,7363,120.401965,120,one-sample z test (one-sided),mean > mu0,1.875055,0.030393,0.05,Reject H0


#### NOTE:

#### What is hypothesis testing?
👉 It is a way to check if a claim is likely true or not using data.

*Example:*
**“Is the average height really 170 cm, or is it different?”**

#### A. How to answer these questions (STEP-BY-STEP)

#### Question:

> Is average systolic BP higher than 120?

#### Step 1: Decide one-sided or two-sided

* “**Higher than**” → **one-sided**

✍️ Write:

> This is a one-sided test.

---

#### Step 2: Write hypotheses (already given)

* **H0**: μ = 120 (no increase)
* **H1**: μ > 120 (higher)

---

#### Step 3: Compute test statistic (z)

**What is z?**
👉 How far the sample mean is from 120, in SE units.

✍️ Formula:

```
z = (sample mean − 120) / SE
```

---

#### Step 4: Compute p-value

**What is p-value?**
👉 Chance of seeing this result **if H0 were true**.

* Small p-value → result is surprising → reject H0
* Big p-value → result is normal → keep H0

---

#### Step 5: Compare with α = 0.05

* p < 0.05 → **Reject H0**
* p ≥ 0.05 → **Fail to reject H0**

---

#### Step 6: Write conclusion (MOST IMPORTANT)

#### If p < 0.05:

> We reject H0 and conclude average systolic BP is higher than 120.

#### If p ≥ 0.05:

> We fail to reject H0 and do not have enough evidence that average systolic BP is higher than 120.

---

#### B. All terms explained (VERY SIMPLE)

#### Mean (μ or x̄)

Average value.

**Example:**
BP = 120, 130, 140 → mean = 130

---

#### Null hypothesis (H0)

What we **assume first**.

**Example:**
Average BP = 120

---

#### Alternative hypothesis (H1)

What we want to **check**.

**Example:**
Average BP > 120

---

#### One-sided test

Only checking **one direction**.

**Example:**
Higher than 120 ✔
Lower than 120 ✔

---

#### Two-sided test

Checking **both directions**.

**Example:**
Different from 120

---

#### Standard Error (SE)

How **uncertain** the mean is.

**Example:**
More people → smaller SE

---

#### Test statistic (z)

How many **SEs away** the mean is from 120.

**Big z** → strong evidence
**Small z** → weak evidence

---

#### p-value

Chance that the result happened **by luck**.

**Example:**
p = 0.01 → very unlikely by luck
p = 0.50 → very likely by luck

---

#### Significance level (α = 0.05)

Our **decision rule**.

👉 We accept being wrong **5 times out of 100**.

---

#### C. One-line memory trick 🧠

```
Question → One or two sided?
Compute z → Get p-value
p < 0.05 → Reject H0
p ≥ 0.05 → Fail to reject H0
```

##### ✅ Task for students: Hypothesis testing of large-sample mean 

Now test whether **average systolic BP** is **different from 120 mm Hg**.

Set up:
- **H0:** μ = 120  
- **H1:** μ ≠ 120 

Using a **large-sample mean test (normal / z test)**:

1) define wheather it is one-sided or two-sided test.
2) compute the test statistic  
3) compute the p-value  
4) state your conclusion at α = 0.05


In [5]:
# Your answer here
Answer = "it is two-sided since the alternative hypothesis is μ ≠ 120 so μ can be less or greater than 120 in the alternative hypothesis"

# Your code here
mu0 = 120

ds = DescrStatsW(x_large)

z_stat, p_value = ds.ztest_mean(value=mu0, alternative="two-sided")

pd.DataFrame([{
    "n": int(ds.nobs),
    "sample_mean": float(ds.mean),
    "mu0": mu0,
    "test": "one-sample z test (two-sided)",
    "alternative": "mean != mu0",
    "z_stat": float(z_stat),
    "p_value": float(p_value),
    "alpha": 0.05,
    "decision": "Reject H0" if p_value < 0.05 else "Fail to reject H0",

}])

,n,sample_mean,mu0,test,alternative,z_stat,p_value,alpha,decision
0,7363,120.401965,120,one-sample z test (two-sided),mean != mu0,1.875055,0.060785,0.05,Fail to reject H0


##### ✅ Task for students:
Create a **small sample** of systolic BP values (n = 15) called `x_small`.

- Use `np.random.default_rng(42)`
- Sample **without replacement**
- (We’ll use this sample for the “small sample” tests.)

In [6]:
# Your code here
rng = np.random.default_rng(42)
x_small = rng.choice(x_large, size=15, replace=False)

x_small, x_small.shape

(array([151.33333333, 136.66666667, 102.66666667, 107.33333333,
        159.33333333, 116.        , 126.66666667, 123.33333333,
         94.66666667, 117.33333333, 101.33333333,  97.33333333,
        106.66666667, 130.        , 114.        ]),
 (15,))

##### ✅ Task for students: Hypothesis testing of small-sample mean (one-sided)

Using your `x_small` sample (n = 15), test whether mean systolic BP is **higher than 120 mm Hg**.

Set up:
- **H0:** μ = 120  
- **H1:** μ > 120  (one-sided)

Use a **small-sample mean test (t test)** with `DescrStatsW`:

1) compute the t statistic  
2) compute the p-value  
3) state your conclusion at α = 0.05


In [7]:
# Your code here
mu0 = 120

ds_small = DescrStatsW(x_small)
t_stat, p_value, dfree = ds_small.ttest_mean(value=mu0, alternative="larger")

pd.DataFrame([{
    "n": int(ds_small.nobs),
    "sample_mean": float(ds_small.mean),
    "mu0": mu0,
    "test": "one-sample t test (one-sided)",
    "alternative": "mean > mu0",
    "t_stat": float(t_stat),
    "df": float(dfree),
    "p_value": float(p_value),
    "alpha": 0.05,
    "conclusion": "Reject H0" if p_value < 0.05 else "Fail to reject H0",
}])

,n,sample_mean,mu0,test,alternative,t_stat,df,p_value,alpha,conclusion
0,15,118.977778,120,one-sample t test (one-sided),mean > mu0,-0.206453,14.0,0.580295,0.05,Fail to reject H0


##### ✅ Task for students: Hypothesis testing of small-sample mean (two-sided)

Using your `x_small` sample (n = 15), test whether mean systolic BP is **different from 120 mm Hg**.

Set up:
- **H0:** μ = 120  
- **H1:** μ ≠ 120  (two-sided)

Use a **small-sample mean test (t test)**:

1) compute the t statistic  
2) compute the p-value  
3) state your conclusion at α = 0.05

In [8]:
# Your code here

mu0 = 120

ds_small = DescrStatsW(x_small)
t_stat, p_value, dfree = ds_small.ttest_mean(value=mu0, alternative="two-sided")

pd.DataFrame([{
    "n": int(ds_small.nobs),
    "sample_mean": float(ds_small.mean),
    "mu0": mu0,
    "test": "one-sample t test (two-sided)",
    "alternative": "mean != mu0",
    "t_stat": float(t_stat),
    "df": float(dfree),
    "p_value": float(p_value),
    "alpha": 0.05,
    "conclusion": "Reject H0" if p_value < 0.05 else "Fail to reject H0",
}])


,n,sample_mean,mu0,test,alternative,t_stat,df,p_value,alpha,conclusion
0,15,118.977778,120,one-sample t test (two-sided),mean != mu0,-0.206453,14.0,0.839409,0.05,Fail to reject H0


##### ✅ Task for students: Hypothesis testing of difference of two large-sample means (one-sided)

Now compare mean systolic BP between two groups:

- Group 1: `RIAGENDR == 1` (Male)
- Group 0: `RIAGENDR == 2` (Female)

We want to test whether males have **higher** average systolic BP.

Set up:
- **H0:** μ1 − μ0 = 0  
- **H1:** μ1 − μ0 > 0  (one-sided)

Use a **two-sample z test** (normal approximation) with `statsmodels.stats.weightstats.ztest`
and `usevar="unequal"`.

Report:
1) the z statistic  
2) the p-value  
3) your conclusion at α = 0.05


In [9]:
# Your code here

g1 = df.loc[df["RIAGENDR"] == 1, "sys_mean"].dropna().astype(float).to_numpy()  # Male
g0 = df.loc[df["RIAGENDR"] == 2, "sys_mean"].dropna().astype(float).to_numpy()  # Female

z_stat, p_value = ztest(g1, g0, value=0, alternative="larger", usevar="unequal")

pd.DataFrame([{
    "n_group1": len(g1),
    "n_group0": len(g0),
    "mean_group1": float(np.mean(g1)),
    "mean_group0": float(np.mean(g0)),
    "test": "two-sample z test (one-sided)",
    "alternative": "mean1 - mean0 > 0",
    "z_stat": float(z_stat),
    "p_value": float(p_value),
    "alpha": 0.05,
    "conclusion": "Reject H0" if p_value < 0.05 else "Fail to reject H0",
}])


,n_group1,n_group0,mean_group1,mean_group0,test,alternative,z_stat,p_value,alpha,conclusion
0,3599,3764,121.97666,118.896298,two-sample z test (one-sided),mean1 - mean0 > 0,7.216231,2.672408e-13,0.05,Reject H0


---

### Part B — Different dataset for Task 6 (RAND HIE)

For Task 6 we will use the **RAND Health Insurance Experiment** dataset bundled with `statsmodels`.

We’ll test a **two-sided** difference in means using **small samples** from two groups.


In [10]:
# Load RAND HIE dataset (bundled with statsmodels)
rand = sm.datasets.randhie.load_pandas().data.copy()
rand[["mdvis", "idp"]].head()

,mdvis,idp
0,0,1
1,2,1
2,0,1
3,0,1
4,0,1


##### ✅ Task for students: Hypothesis testing of difference of two small-sample means (two-sided)

We will compare **doctor visits** (`mdvis`) between two groups in the RAND HIE data:

- Group 1: `idp == 1`
- Group 0: `idp == 0`

To make this a **small-sample** exercise:
- Randomly sample **n = 20** observations from each group (without replacement, `rng = np.random.default_rng(7)`).

Set up:
- **H0:** μ1 − μ0 = 0  
- **H1:** μ1 − μ0 ≠ 0  (two-sided)

Use a **two-sample t test** with `statsmodels.stats.weightstats.ttest_ind`
and `usevar="unequal"`.

Report:
1) the t statistic  
2) the p-value  
3) your conclusion at α = 0.05


In [11]:
rng = np.random.default_rng(7)

g1_all = rand.loc[rand["idp"] == 1, "mdvis"].dropna().astype(float).to_numpy()
g0_all = rand.loc[rand["idp"] == 0, "mdvis"].dropna().astype(float).to_numpy()

g1 = rng.choice(g1_all, size=20, replace=False)
g0 = rng.choice(g0_all, size=20, replace=False)


# Your code here
t_stat, p_value, dfree = ttest_ind(g1, g0, alternative="two-sided", usevar="unequal", value=0)

pd.DataFrame([{
    "n_group1": len(g1),
    "n_group0": len(g0),
    "mean_group1": float(np.mean(g1)),
    "mean_group0": float(np.mean(g0)),
    "test": "two-sample t test (two-sided)",
    "alternative": "mean1 - mean0 != 0",
    "t_stat": float(t_stat),
    "df": float(dfree),
    "p_value": float(p_value),
    "alpha": 0.05,
    "conclusion": "Reject H0" if p_value < 0.05 else "Fail to reject H0",
}])


,n_group1,n_group0,mean_group1,mean_group0,test,alternative,t_stat,df,p_value,alpha,conclusion
0,20,20,2.65,5.85,two-sample t test (two-sided),mean1 - mean0 != 0,-1.783655,34.161348,0.083366,0.05,Fail to reject H0


#### NOTE: Some related terms in short

---

#### z-test

**What:** Test using **normal (Z)** distribution
**When:** Large sample (n ≥ 30)
**Why:** Big samples are stable

**Example:**
Test if average BP > 120 using 500 people

---

#### t-test

**What:** Test using **t distribution**
**When:** Small sample (n < 30)
**Why:** Small samples are uncertain

**Example:**
Test if average BP > 120 using 15 people

---

#### dfree (degrees of freedom)

**What:** How many values can change freely
**Formula:** df = n − 1
**Why:** T-test needs it to adjust uncertainty

**Example:**
n = 15 → df = 14

---

#### alpha (α)

**What:** Error limit (usually 0.05)
**Why:** Rule for decision

**Meaning:**
We allow **5% chance** of being wrong

---

#### p-value

**What:** Chance result happened by luck
**Why:** To decide

* p < α → Reject H0
* p ≥ α → Fail to reject H0

---

#### H0 (null hypothesis)

**What:** Default assumption
**Example:** Mean BP = 120

---

#### H1 (alternative hypothesis)

**What:** What we want to prove
**Example:** Mean BP > 120

---

#### One-line memory rule 🧠

```
Big sample → z-test
Small sample → t-test
df = n − 1
p < 0.05 → Reject
```